# Climate Change and Global Temperature Analysis

Student Project
Dataset: Berkeley Earth Global Temperature Data

In [ ]:
# Cell 2: Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

print('All libraries imported successfully')

In [ ]:
# Cell 3: Load Datasets
global_temp = pd.read_csv('GlobalTemperatures.csv')
country_temp = pd.read_csv('GlobalLandTemperaturesByCountry.csv')

global_temp['dt'] = pd.to_datetime(global_temp['dt'])
country_temp['dt'] = pd.to_datetime(country_temp['dt'])

global_temp['Year'] = global_temp['dt'].dt.year
country_temp['Year'] = country_temp['dt'].dt.year

print('Global Temperature Shape:', global_temp.shape)
print('Country Temperature Shape:', country_temp.shape)
print('')
global_temp.head()

In [ ]:
# Cell 4: Data Cleaning
global_clean = global_temp.dropna(subset=['LandAverageTemperature']).copy()
country_clean = country_temp.dropna(subset=['AverageTemperature']).copy()

yearly_avg = global_clean.groupby('Year')['LandAverageTemperature'].mean().reset_index()
yearly_avg.columns = ['Year', 'AvgTemp']

baseline = yearly_avg[(yearly_avg['Year'] >= 1900) & (yearly_avg['Year'] <= 1950)]['AvgTemp'].mean()

yearly_avg['Anomaly'] = yearly_avg['AvgTemp'] - baseline
yearly_avg['Rolling10'] = yearly_avg['AvgTemp'].rolling(window=10).mean()

print('Data cleaned successfully')
print('Baseline Temperature (1900-1950):', round(baseline, 2), 'C')
print('')

In [ ]:
# Cell 5: Chart 1 - Global Temperature Trend
# I am plotting the long term trend of global temperature
plt.figure()
plt.plot(yearly_avg['Year'], yearly_avg['AvgTemp'], color='lightblue', label='Yearly Average')
plt.plot(yearly_avg['Year'], yearly_avg['Rolling10'], color='red', linewidth=2.5, label='10 Year Rolling Average')

plt.title('Global Average Land Temperature Over Time')
plt.xlabel('Year')
plt.ylabel('Temperature (C)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Cell 6: Chart 2 - Temperature Anomaly
# This chart shows how much warmer or cooler each year is compared to the baseline
recent = yearly_avg[yearly_avg['Year'] >= 1850]

plt.figure()
colors = ['red' if x > 0 else 'blue' for x in recent['Anomaly']]

plt.bar(recent['Year'], recent['Anomaly'], color=colors, alpha=0.8)
plt.axhline(y=0, color='black', linestyle='--')

plt.title('Temperature Anomaly from 1900-1950 Baseline')
plt.xlabel('Year')
plt.ylabel('Anomaly (C)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Cell 7: Chart 3 - Top 10 Hottest and Coldest Countries
# Comparing hottest and coldest countries using recent data
modern = country_clean[country_clean['Year'] >= 2000]
country_avg = modern.groupby('Country')['AverageTemperature'].mean().reset_index()

top10_hot = country_avg.nlargest(10, 'AverageTemperature')
top10_cold = country_avg.nsmallest(10, 'AverageTemperature')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(top10_hot['Country'], top10_hot['AverageTemperature'], color='salmon')
axes[0].set_title('Top 10 Hottest Countries (2000 onwards)')
axes[0].set_xlabel('Average Temperature (C)')

axes[1].barh(top10_cold['Country'], top10_cold['AverageTemperature'], color='skyblue')
axes[1].set_title('Top 10 Coldest Countries (2000 onwards)')
axes[1].set_xlabel('Average Temperature (C)')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Chart 4 - Average Temperature by Decade
# This shows the average temperature in each decade
yearly_avg['Decade'] = (yearly_avg['Year'] // 10) * 10
decade_avg = yearly_avg[yearly_avg['Year'] >= 1850].groupby('Decade')['AvgTemp'].mean().reset_index()

plt.figure()
plt.bar(decade_avg['Decade'].astype(str), decade_avg['AvgTemp'], color='orange', alpha=0.8)

plt.title('Average Global Temperature by Decade')
plt.xlabel('Decade')
plt.ylabel('Average Temperature (C)')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.show()

In [ ]:
# Cell 9: Chart 5 - Monthly Average Temperature
# Checking seasonal pattern across all years
monthly_avg = global_clean.groupby('Month')['LandAverageTemperature'].mean().reset_index() if 'Month' in global_clean.columns else global_clean.copy()

# Add Month column if not present
global_clean['Month'] = global_clean['dt'].dt.month
monthly_avg = global_clean.groupby('Month')['LandAverageTemperature'].mean().reset_index()

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

plt.figure(figsize=(10, 5))
sns.barplot(x='Month', y='LandAverageTemperature', data=monthly_avg, palette='coolwarm')
plt.xticks(ticks=range(12), labels=month_names)
plt.title('Average Temperature by Month')
plt.ylabel('Temperature (C)')
plt.show()

In [ ]:
# Cell 10: Chart 6 - Box Plot by Century (New)
# I want to see how temperature distribution changed across different centuries
global_clean['Century'] = (global_clean['Year'] // 100) * 100

# Mapping century numbers to labels
century_labels = {1700: '1700s', 1800: '1800s', 1900: '1900s', 2000: '2000s'}
global_clean['Century_Label'] = global_clean['Century'].map(century_labels)

# Removing rows where century label is missing
global_clean = global_clean.dropna(subset=['Century_Label'])

plt.figure(figsize=(10, 6))
sns.boxplot(x='Century_Label', y='LandAverageTemperature',
            data=global_clean, palette='coolwarm',
            order=['1700s', '1800s', '1900s', '2000s'])

plt.title('Temperature Distribution by Century')
plt.xlabel('Century')
plt.ylabel('Land Average Temperature (C)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Cell 11: Interactive World Map
world_avg = country_clean[country_clean['Year'] >= 2000].groupby('Country')['AverageTemperature'].mean().reset_index()

fig = px.choropleth(
    world_avg,
    locations='Country',
    locationmode='country names',
    color='AverageTemperature',
    color_continuous_scale='RdYlBu_r',
    title='Average Land Temperature by Country (2000-2015)',
    height=600
)
fig.show()

In [ ]:
# Cell 12: Summary
print('Project Summary')
print('')

total_warming = yearly_avg[yearly_avg['Year'] >= 2000]['AvgTemp'].mean() - \
                yearly_avg[yearly_avg['Year'].between(1850, 1900)]['AvgTemp'].mean()

hottest_year = yearly_avg.loc[yearly_avg['AvgTemp'].idxmax()]

print('Total warming since 1850s:', round(total_warming, 2), 'C')
print('Hottest year on record:', int(hottest_year['Year']))
print('Number of countries analyzed:', country_clean['Country'].nunique())
print('')

print('Key Observations:')
print('1. Global temperatures are clearly increasing over time.')
print('2. The warming speed has increased a lot after 1980.')
print('3. Different centuries show higher temperatures in recent times.')
print('4. Some countries are much hotter or colder than others.')

print('')
print('Conclusion:')
print('From this analysis, I can see that the Earth is getting warmer.')
print('The box plot shows that the 2000s are noticeably warmer than previous centuries.')
print('This project helped me understand the reality of climate change through data.')